In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from my_utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

##### Agent Creation

In [4]:
# Senior Data Engineer Agent
data_engineer = Agent(
    role = "Senior Data Engineer",
    goal = """ Connect to MySQL database and extract clean,
            complete customer records that will be used for
            churn analysis and model training or prediction,""",

    backstory = ("You are a seasoned Data Engineer with 8 years of"
                "experience building data pipelines for enterprise"
                "companies. You specialize in MySQL and python integration,"
                "writing optimized SQL queries that extract exactly the right data"
                "with zero redundancy. You are obsessed with data quality,"
                "you never pass dirty, incomplete, or malformed data downstream."
                "Your colleagues trust your extractions completely."),
        
    allow_delegation = False,
    verbose = True

)

In [5]:
#Senior Data Scientist Agent
data_scientist = Agent(
    role="Senior Data Scientist",

    goal="""Transform the raw customer data into a clean,
            well-engineered feature set by encoding categorical
            columns and ensuring all numerical columns are
            correctly formatted and ready for model training
            or inference.""",

    backstory=("""You are a Senior Data Scientist with deep expertise
                 in customer behaviour analytics for telecom and
                 subscription businesses. You have built churn models
                 for companies across Nigeria, Kenya and South Africa.
                 You understand that the quality of features determines
                 the quality of predictions. You know exactly which
                 columns need encoding, which are numerical signals,
                 and how to structure data so a machine learning model
                 can extract maximum predictive power from it."""),

    allow_delegation=False,
    verbose=True
)

In [6]:
#Senior ML Engineer Agent
ml_engineer = Agent(
    role="Senior Machine Learning Engineer",
    
    goal="""Train a robust, accurate Random Forest churn 
            prediction model on the engineered features, 
            evaluate its performance thoroughly, and save 
            it to disk for future weekly predictions.""",
    
    backstory=("""You are a Senior ML Engineer who has 
                 productionized over 30 machine learning 
                 models across financial services and 
                 telecommunications industries. You treat 
                 every model like it is going to production 
                 on day one — you always evaluate with 
                 classification reports and ROC-AUC scores, 
                 handle class imbalance carefully, and never 
                 save a model you haven't validated. You 
                 believe a saved model is a promise to the 
                 business."""),
    
    allow_delegation=False,
    verbose=True
)

In [7]:
#Senior Churn Analyst Agent
churn_analyst = Agent(
    role="Senior Churn Analyst",
    
    goal="""Load the trained churn model, score this week's 
            customers, identify HIGH and MEDIUM risk customers, 
            and send a clear, actionable alert email to the 
            business team so they can take immediate retention 
            action.""",
    
    backstory="""You are a Senior Churn Analyst who sits at 
                 the intersection of data science and business 
                 strategy. You have saved companies millions 
                 in lost revenue by catching at-risk customers 
                 early and triggering the right retention 
                 actions. You are precise — you only flag 
                 customers with genuine churn signals and 
                 never cry wolf with false positives. Your 
                 alert emails are clean, direct, and always 
                 tell the business team exactly who is at 
                 risk, how serious it is, and what they 
                 should do next.""",
    
    allow_delegation=False,
    verbose=True
)

##### Creating the Tools

In [8]:
#loading all nessary libraries
import os
import pickle
import smtplib
import argparse
import pandas as pd
import numpy as np
import mysql.connector
from dotenv import load_dotenv


from io import StringIO
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from crewai_tools import tool

from apscheduler.schedulers.blocking import BlockingScheduler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder

In [9]:
# Database connection function
load_dotenv()
def get_connection():
    return mysql.connector.connect(
        port=3306,
       host = os.getenv('DB_HOST'),
       user = os.getenv('DB_USER'),
       password = os.getenv('DB_PASSWORD'),
       database = os.getenv('DB_NAME')
    )

#TOOL 1: Fetch Customer Data
@tool("Fetch Customer Data")
def fetch_data(mode: str) -> str:
    """
    Connects to the MySQL database and fetches customer
    records from the customer_data table. Saves the data
    to a temp CSV file and returns the file path.
    """
    conn = get_connection()
    query = "SELECT * FROM customer_data"
    df = pd.read_sql(query, conn)
    conn.close()

    os.makedirs("temp", exist_ok=True)
    path = "temp/raw_data.csv"
    df.to_csv(path, index=False)

    print(f"✅ Fetched {len(df)} records — saved to {path}")
    return path  # ← return path not data

In [10]:
# TOOL 2: Engineer Features
@tool("Engineer Features")
def engineer_features(file_path: str) -> str:
    """
    Reads raw customer data from the given file path,
    encodes categorical columns Gender, Subscription_Type
    and Contract_Type using Label Encoding, keeps all
    numerical columns as-is. Saves processed features
    to a temp file and returns the file path.
    """
    df = pd.read_csv(file_path)  # ← read from file

    categorical_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod']
    le = LabelEncoder()

    for col in categorical_cols:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))

    numerical_cols = [
       'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'
    ]

    feature_cols = categorical_cols + numerical_cols

    # 🔥 Robust ID detection
    possible_ids = ["CustomerID", "customerID", "customer_id", "cust_id", "id"]

    customer_col = None
    for col in possible_ids:
        if col in df.columns:
            customer_col = col
            break

    if customer_col is None:
        raise ValueError(f"No customer ID column found. Available columns: {list(df.columns)}")

    # Keep ID ALWAYS
    if "Churn" in df.columns:
        result = df[[customer_col] + feature_cols + ["Churn"]]
    else:
        result = df[[customer_col] + feature_cols]

    path = "temp/engineered_data.csv"
    result.to_csv(path, index=False)

    print(f"✅ Feature engineering complete — saved to {path}")
    print(f"📌 Columns: {result.columns.tolist()}")

    return path

In [11]:
#Tool 3: Train Model
@tool("Train Churn Model")
def train_model(file_path: str) -> str:
    """
    Reads engineered features from the given file path
    and trains a Random Forest classifier to predict
    customer churn. Saves trained model to
    models/churn_model.pkl. Returns a performance summary.
    """
    df = pd.read_csv(file_path)

    feature_cols = [
       'gender', 'Partner', 'Dependents', 'PhoneService',
       'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaperlessBilling', 'PaymentMethod',
       'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'
    ]

    X = df[feature_cols]
    y = df["Churn"]

    # Step 1 — Split first
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Step 2 — Apply SMOTE on training data only
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(
        X_train, y_train
    )

    print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
    print(f"After SMOTE : {y_train_resampled.value_counts().to_dict()}")

    # Step 3 — Train on resampled data
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        random_state=42,
        class_weight="balanced"
    )
    model.fit(X_train_resampled, y_train_resampled)

    # Step 4 — Evaluate on ORIGINAL test data (not resampled)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    report  = classification_report(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    os.makedirs("models", exist_ok=True)
    with open("models/churn_model.pkl", "wb") as f:
        pickle.dump(model, f)

    summary = f"""
✅ Model Training Complete
===========================
ROC-AUC Score : {roc_auc:.4f}
Model saved to: models/churn_model.pkl

Classification Report:
{report}
"""
    print(summary)
    return summary

In [12]:
# Tool 4: Predict and Score Customers
@tool("Predict and Score Churn")
def predict_and_score(file_path: str) -> str:
    """
    Reads engineered features from the given file path,
    loads the saved churn model and scores all customers.
    Classifies into HIGH, MEDIUM or LOW risk tiers.
    Saves results and returns file path of at-risk customers.
    """
    df = pd.read_csv(file_path)

    with open("models/churn_model.pkl", "rb") as f:
        model = pickle.load(f)

    feature_cols = [
        'gender', 'Partner', 'Dependents', 'PhoneService',
        'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
        'Contract', 'PaperlessBilling', 'PaymentMethod',
        'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'
    ]

    # Detect customer ID column (FIX: proper indentation)
    customer_col = "CustomerID" if "CustomerID" in df.columns else "customerID"

    X = df[feature_cols]
    churn_probs = model.predict_proba(X)[:, 1]
    df["churn_probability"] = np.round(churn_probs, 4)

    def assign_tier(prob):
        if prob >= 0.75:
            return "HIGH"
        elif prob >= 0.50:
            return "MEDIUM"
        else:
            return "LOW"

    df["risk_tier"] = df["churn_probability"].apply(assign_tier)

    at_risk = df[df["risk_tier"].isin(["HIGH", "MEDIUM"])].copy()
    at_risk = at_risk.sort_values("churn_probability", ascending=False)

    path = "temp/at_risk.csv"
    at_risk[[customer_col, "churn_probability", "risk_tier"]].to_csv(
        path, index=False
    )

    print(f"✅ Identified {len(at_risk)} at-risk customers")
    print(f"   🔴 HIGH   : {len(at_risk[at_risk['risk_tier'] == 'HIGH'])}")
    print(f"   🟡 MEDIUM : {len(at_risk[at_risk['risk_tier'] == 'MEDIUM'])}")

    return path

In [13]:
# Tool 5: Send Alert Email

def format_message(df: pd.DataFrame) -> str:
    """Format the at-risk customers into a readable email body."""
    high_risk = len(df[df['risk_tier'] == 'HIGH'])
    medium_risk = len(df[df['risk_tier'] == 'MEDIUM'])
    
    body = f"""
Monthly Churn Alert — {datetime.now().strftime('%B %Y')}

ALERT SUMMARY:
🔴 HIGH RISK CUSTOMERS: {high_risk}
🟡 MEDIUM RISK CUSTOMERS: {medium_risk}

Total At-Risk: {len(df)}

Details:
{df.to_string(index=False)}

Action Required: Contact these customers immediately for retention.
"""
    return body
    
@tool("Send Churn Alert Email")
def send_alert(file_path: str) -> str:
    """
    Reads at-risk customers from the given file path,
    formats a monthly churn alert email and sends it
    via Gmail SMTP. Returns confirmation string.
    """
    df = pd.read_csv(file_path)

    sender    = os.getenv("ALERT_EMAIL_SENDER")
    recipient = os.getenv("ALERT_EMAIL_RECIPIENT")
    password  = os.getenv("ALERT_EMAIL_PASSWORD")

    body = format_message(df)

    # Check if credentials exist
    if not sender or not recipient or not password:
        print("\n⚠️  EMAIL CREDENTIALS MISSING")
        print(f"   ALERT_EMAIL_SENDER: {'✓ Set' if sender else '✗ Not set'}")
        print(f"   ALERT_EMAIL_RECIPIENT: {'✓ Set' if recipient else '✗ Not set'}")
        print(f"   ALERT_EMAIL_PASSWORD: {'✓ Set' if password else '✗ Not set'}")
        print(f"\n📧 Email would be sent to: {recipient or 'NOT CONFIGURED'}")
        print(f"📊 At-risk customers: {len(df)}")
        confirmation = f"✅ Alert prepared (credentials missing) — {len(df)} at-risk customers"
        print(f"\n{confirmation}\n")
        return confirmation

    try:
        email = MIMEMultipart()
        email["From"]    = sender
        email["To"]      = recipient
        email["Subject"] = "⚠️ Monthly Churn Alert — Action Required"
        email.attach(MIMEText(body, "plain"))

        print("\n📧 Connecting to Gmail SMTP server...")
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(sender, password)
            print(f"✅ Login successful as {sender}")
            server.sendmail(sender, recipient, email.as_string())
            print(f"✅ Email sent successfully to {recipient}")

        confirmation = f"✅ Alert sent to {recipient} — {len(df)} at-risk customers"
        print(f"\n{confirmation}\n")
        return confirmation

    except smtplib.SMTPAuthenticationError:
        print("\n❌ EMAIL AUTHENTICATION FAILED")
        print(f"   Check your email password or use an App Password")
        print(f"   For Gmail: https://myaccount.google.com/apppasswords\n")
        return f"❌ Email send failed — authentication error"

    except Exception as e:
        print(f"\n❌ EMAIL SEND FAILED: {str(e)}\n")
        return f"❌ Email send failed — {str(e)}"

##### Creating Tasks


In [14]:
#Training Fetch Task
train_fetch_task = Task(
    description=("""Connect to the MySQL database and fetch all 
                   customer records including the Churn label. 
                   Use the Fetch Customer Data tool with mode 'train'.
                   Your FINAL ANSWER must be ONLY the file path 
                   returned by the tool. Do not generate or return 
                   any data yourself."""),

    expected_output=("""Only the file path string returned by the 
                       Fetch Customer Data tool. 
                       Example: temp/raw_data.csv"""),

    tools = [fetch_data],
    agent = data_engineer
)

#Training Engineer task
train_engineer_task = Task(
    description=("""Use the Engineer Features tool with the file path 
                   from the previous task which is temp/raw_data.csv.
                   Encode categorical columns Gender, Subscription_Type 
                   and Contract_Type. Keep all numerical columns as-is. 
                   Your FINAL ANSWER must be ONLY the file path 
                   returned by the tool. Do not generate any data yourself."""),

    expected_output=("""Only the file path string returned by the 
                       Engineer Features tool.
                       Example: temp/engineered_data.csv"""),
    tools = [engineer_features],
    agent = data_scientist,
    context = [train_fetch_task]
)

train_model_task = Task(
    description=("""Use the Train Churn Model tool with the file path 
                   from the previous task which is temp/engineered_data.csv.
                   Train the Random Forest model and save it to disk.
                   Your FINAL ANSWER must be ONLY the performance 
                   summary returned by the tool."""),

    expected_output=("""The model performance summary showing ROC-AUC 
                       score, classification report and confirmation 
                       that model was saved to models/churn_model.pkl"""),

    tools=[train_model],
    agent=ml_engineer,
    context=[train_engineer_task]
)


# --- PREDICTION TASKS ---

predict_fetch_task = Task(
    description=("""Connect to the MySQL database and fetch all 
                   customer records for monthly churn scoring.
                   Use the Fetch Customer Data tool with mode 'predict'.
                   Your FINAL ANSWER must be ONLY the file path 
                   returned by the tool. Do not generate any data yourself."""),

    expected_output=("""Only the file path string returned by the 
                       Fetch Customer Data tool.
                       Example: temp/raw_data.csv""")   ,

    tools = [fetch_data],
    agent=data_engineer
)

predict_engineer_task = Task(
    description=("""Use the Engineer Features tool with the file path 
                   from the previous task which is temp/raw_data.csv.
                   Encode categorical columns Gender, Subscription_Type 
                   and Contract_Type. Keep CustomerID for identification.
                   Your FINAL ANSWER must be ONLY the file path 
                   returned by the tool. Do not generate any data yourself."""),

    expected_output=("""Only the file path string returned by the 
                       Engineer Features tool.
                       Example: temp/engineered_data.csv"""),

    tools=[engineer_features],
    agent=data_scientist,
    context=[predict_fetch_task]
)

predict_score_task = Task(
    description=("""IMPORTANT: You MUST do TWO things in order:
    
    1) Use the Predict and Score Churn tool with file path temp/engineered_data.csv
       to score all customers and identify HIGH/MEDIUM risk customers.
       This will create temp/at_risk.csv
    
    2) THEN use the Send Churn Alert Email tool with file path temp/at_risk.csv
       to send the alert email to the business team.
    
    Your FINAL ANSWER must be the email confirmation message."""),

    expected_output=("""Confirmation that the alert email was sent 
                       successfully with the number of HIGH and MEDIUM 
                       risk customers identified."""),
                       
    tools=[predict_and_score, send_alert],
    agent=churn_analyst,
    context=[predict_engineer_task]
)

##### Creating Crews

In [15]:
training_crew = Crew(
    agents =[data_engineer, data_scientist, ml_engineer],
    tasks=[train_fetch_task, train_engineer_task, train_model_task],
    processing_mode="sequential",
    verbose=True
)

prediction_crew = Crew(
    agents=[data_engineer, data_scientist, churn_analyst],
    tasks=[predict_fetch_task, predict_engineer_task, predict_score_task],
    processing_mode="sequential",
    verbose=True
)

2026-04-15 11:14:13,423 - 16904 - __init__.py-__init__:567 - WARNING: Overriding of current TracerProvider is not allowed


#### Scheduler - Runs Prediction Crew Monthly


In [16]:
def run_monthly_prediction():
    print("\n🚀 Monthly Churn Prediction Started...")
    prediction_crew.kickoff()
    print("\n✅ Monthly Churn Prediction Complete")

##### Main Entry Point

In [17]:
def main(mode: str):
    if mode == "train":
        print("\n🚀 Training Mode Started...")
        training_crew.kickoff()
        print("\n✅ Training Complete — model saved to models/churn_model.pkl")

    elif mode == "predict":
        print("\n🚀 Running One-Time Prediction...")
        run_monthly_prediction()

    elif mode == "schedule":
        print("\n📅 Scheduler Started — prediction runs on the 1st of every month at 8:00 AM")
        print("   Press Ctrl+C to stop\n")

        scheduler = BlockingScheduler()
        scheduler.add_job(
            run_monthly_prediction,
            trigger="cron",
            day=1,
            hour=8,
            minute=0
        )

        try:
            scheduler.start()
        except (KeyboardInterrupt, SystemExit):
            print("\n⛔ Scheduler stopped")

In [18]:
main("train") # to train the model for the first time


🚀 Training Mode Started...
 [DEBUG]: == Working Agent: Senior Data Engineer
 [INFO]: == Starting Task: Connect to the MySQL database and fetch all 
                   customer records including the Churn label. 
                   Use the Fetch Customer Data tool with mode 'train'.
                   Your FINAL ANSWER must be ONLY the file path 
                   returned by the tool. Do not generate or return 
                   any data yourself.


> Entering new CrewAgentExecutor chain...
I need to connect to the MySQL database and fetch all customer records including the Churn label.

Action: Fetch Customer Data
Action Input: {"mode": "train"}✅ Fetched 7032 records — saved to temp/raw_data.csv
 

temp/raw_data.csv

Final Answer: temp/raw_data.csv

> Finished chain.
 [DEBUG]: == [Senior Data Engineer] Task output: temp/raw_data.csv


 [DEBUG]: == Working Agent: Senior Data Scientist
 [INFO]: == Starting Task: Use the Engineer Features tool with the file path 
                   fr

In [19]:
main("predict") # to run a one-time prediction and send alert email


🚀 Running One-Time Prediction...

🚀 Monthly Churn Prediction Started...
 [DEBUG]: == Working Agent: Senior Data Engineer
 [INFO]: == Starting Task: Connect to the MySQL database and fetch all 
                   customer records for monthly churn scoring.
                   Use the Fetch Customer Data tool with mode 'predict'.
                   Your FINAL ANSWER must be ONLY the file path 
                   returned by the tool. Do not generate any data yourself.


> Entering new CrewAgentExecutor chain...
I need to use the Fetch Customer Data tool with mode 'predict' to fetch all customer records for monthly churn scoring.

Action: Fetch Customer Data
Action Input: {"mode": "predict"}✅ Fetched 7032 records — saved to temp/raw_data.csv
 

temp/raw_data.csv

Final Answer: temp/raw_data.csv

> Finished chain.
 [DEBUG]: == [Senior Data Engineer] Task output: temp/raw_data.csv


 [DEBUG]: == Working Agent: Senior Data Scientist
 [INFO]: == Starting Task: Use the Engineer Features tool w